# App-07-CSharp : Solveur Wordle -- CSP et théorie de l'information (jumeau C#)

> **Twin C#** de [`App-07-Wordle-Python`](App-07-Wordle-Python.ipynb) (Python : `numpy`, `matplotlib`, filtration CSP + entropie).
> Marathon **.NET / Python** (#4956), volet **Search / Applications / CSP**.

**Navigation** : [<< App-7 Python](App-07-Wordle-Python.ipynb) | [Index](../README.md) | [App-8 MiniZinc >>](App-08-MiniZinc-Python.ipynb)

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :

1. **Modeliser** Wordle comme un CSP (variables = positions, domaines = lettres, contraintes = feedback).
2. **Implementer from-scratch** trois familles de solveurs : filtrage aleatoire, propagation de contraintes (CSP), et maximisation d'entropie.
3. **Calculer l'entropie de Shannon** d'un feedback et l'utiliser pour choisir le mot le plus informatif.
4. **Comparer** les trois approches sur un benchmark de mots secrets.

Le code est entierement reecrit en C# (.NET 9, 0 NuGet hors ScottPlot optionnel), en miroir de la version Python qui utilisait `numpy`/`matplotlib`. Les quantites déterministes (entropie des mots, classement des meilleurs premières tentatives) **concordent au bit pres** entre les deux langages : c'est le critere de parite.


In [1]:
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;

// Formatage invariant (la culture FR ne persiste pas entre cellules en .NET Interactive).
CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.DefaultThreadCurrentCulture = CultureInfo.InvariantCulture;

// PRNG reproductible (les resultats aleatoires DIFFERENT du Python : Random != random.seed,
// mais les quantites deterministes -- entropie, top-10 -- concordent exactement, cf. section parite).
var rng = new Random(42);

Console.WriteLine("C# Wordle twin -- marathon #4956. Kernel .net-csharp, 0 NuGet (algorithme from-scratch).");


C# Wordle twin -- marathon #4956. Kernel .net-csharp, 0 NuGet (algorithme from-scratch).


***
## 1. Système de feedback

Wordle code le retour d'une tentative par 5 symboles :

- **Vert (2)** : la lettre est correcte et bien placee ;
- **Jaune (1)** : la lettre est presente dans le mot secret mais a une autre position ;
- **Gris (0)** : la lettre n'est pas presente (ou plus presente que dans la tentative).

Le piege classique concerne les **lettres repeatedes**. Si la tentative contient deux `E` mais le secret un seul, un seul des deux `E` recoit un retour non-gris. L'algorithme ci-dessous resout ceci par une **double passe** : d'abord les verts (en marquant les lettres du secret comme consommees), ensuite les jaunes (en cherchant une occurrence encore libre).


In [2]:
// Liste de mots francais de 5 lettres (sans accents, majuscules).
// STRICTEMENT identique a la WORD_LIST du notebook Python (297 mots) -> parite d'entropie.
static readonly string[] WORD_LIST_SRC = {
    "ABORD","ACIER","ADORE","AGENT","AIDER","AIMER","AINSI","ALBUM","ALGUE","ALLEE","ALLER",
    "AMOUR","ANCRE","ANGLE","ANNEE","APPEL","ARBRE","ARENE","ARGOT","ASTRE","ATLAS","AVENT",
    "AVION","AVOIR","BADGE","BAGUE","BAIES","BALLE","BANDE","BARBE","BARGE","BASER","BATON",
    "BIBLE","BIDON","BILAN","BLANC","BLASE","BOIRE","BONUS","BOTTE","BOURG","BRAVE","BRISE",
    "BRUME","BULLE","CABLE","CADRE","CALME","CANNE","CARTE","CAUSE","CEDER","CHAMP","CHANT",
    "CHASE","CHOIX","CHUTE","CIBLE","CLAIR","CLONE","COEUR","COMTE","CONTE","CORDE","COUDE",
    "COUPE","CRIER","CRIME","CRISE","CRANE","CREUX","CROIX","CRUEL","CYCLE","DANSE","DEBUT","DELTA",
    "DENSE","DEPOT","DOIGT","DOUER","DRAME","DROIT","DUPER","ECLAT","ECRAN","ELEVE","EMAIL",
    "ENVIE","EPAVE","EPICE","ESSAI","ETAGE","EXACT","EXILE","EXTRA","FABLE","FARCE","FAUTE",
    "FERME","FIBRE","FIGER","FILER","FINAL","FLEUR","FOLIE","FORCE","FORME","FORUM","FOSSE",
    "FRAIS","FRANC","FREIN","FRONT","FRUIT","FUMER","GAMIN","GARDE","GELER","GENRE","GLACE",
    "GLOBE","GRADE","GRAIN","GRAND","GRISE","GUIDE","HABIT","HAINE","HERBE","HEURE","HUILE",
    "HYPER","IMAGE","INDEX","ISSUE","JAMBE","JEUNE","JOUER","JUGER","JUSTE","LACET","LAINE",
    "LANCE","LARGE","LASER","LEVER","LIBRE","LIGUE","LINER","LISSE","LITRE","LIVRE","LOGIS",
    "LOUER","LOURD","LUCRE","LUEUR","LYCEE","MACON","MALLE","MARGE","MARIN","MASSE","MENER",
    "MERCI","MEULE","MINCE","MIXTE","MODEM","MONDE","MORAL","MORSE","MOULE","NAPPE","NEIGE",
    "NOBLE","NUAGE","OASIS","OCEAN","OFFRE","OLIVE","OMBRE","ONCLE","OPERA","ORAGE","ORDRE",
    "ORGUE","OTAGE","PAIRE","PANNE","PARIE","PAROI","PARTI","PATIN","PAUSE","PEAGE","PEINE",
    "PELER","PENTE","PERLE","PHARE","PIECE","PINCE","PISTE","PLACE","PLAGE","PLEIN","PLUIE",
    "POCHE","POEME","POINT","POMPE","PORTE","POSER","POSTE","POUCE","PRIME","PRISE","PROSE",
    "PRUNE","QUAND","RAIDE","RAMPE","RANGE","RAVIN","REGAL","REGLE","REINE","REPAS","RESTE",
    "RICHE","RIDER","RIVAL","RONCE","SABLE","SALON","SAUGE","SAUVE","SCENE","SEMER","SERIE",
    "SIGNE","SIROP","SOBRE","SOCLE","SOLDE","SOMME","SONDE","SOUCI","STADE","STYLE","SUITE",
    "SUPER","TABLE","TACHE","TALON","TASSE","TEMPS","TENTE","TERRE","THEME","TIGRE","TITRE",
    "TOILE","TONNE","TOTAL","TRACE","TRAIN","TRAIT","TREVE","TRIBU","TRONC","TUILE","ULTRA",
    "UNION","UNITE","USAGE","USINE","VAGUE","VALSE","VASTE","VEINE","VENTE","VERRE","VERTU",
    "VIDER","VIGNE","VITRE","VIVRE","VOILE","VOLER","VOTER","YACHT","ZEBRA","ZONES"
};

static readonly List<string> WORD_LIST = WORD_LIST_SRC
    .Select(w => w.ToUpperInvariant().Trim())
    .Where(w => w.Length == 5)
    .Distinct().OrderBy(w => w).ToList();

Console.WriteLine($"Liste de mots : {WORD_LIST.Count} mots de 5 lettres");
Console.WriteLine($"Exemples : {string.Join(", ", WORD_LIST.Take(10))}");
Console.WriteLine($"Derniers : {string.Join(", ", WORD_LIST.TakeLast(5))}");


Liste de mots : 297 mots de 5 lettres


Exemples : ABORD, ACIER, ADORE, AGENT, AIDER, AIMER, AINSI, ALBUM, ALGUE, ALLEE


Derniers : VOLER, VOTER, YACHT, ZEBRA, ZONES


### Lecture — 297 mots : la taille du corpus fixe toute l'expérience

La sortie confirme **297 mots de 5 lettres**, sans accents et en majuscules : `ABORD, ACIER, ADORE, AGENT, AIDER, AIMER, AINSI, ALBUM, ALGUE, ALLEE` ouvrent la liste, `VOLER, VOTER, YACHT, ZEBRA, ZONES` la ferment. Quatre conséquences directes :

- **L'espace de recherche est petit** : 297 candidats au départ fixent l'entropie maximale théorique à `log2(297)` bits — le plafond que la sortie de la section 4 retrouve — et rendent exhaustive la stratégie « calculer le feedback de chaque mot » : classer les 297 mots d'ouverture ne demande qu'une seule passe de calcul.
- **Le corpus est trié à la main** : les exemples sont des mots courants, les derniers incluent des emprunts (`ZEBRA`, `YACHT`). La diversité de lettres est recherchée, pas la fréquence d'usage — l'entropie d'un mot mesure donc son pouvoir discriminant DANS cette liste, pas dans le français réel.
- **Pas d'accents ni de cédilles** : `ComputeFeedback` compare des caractères ASCII — la normalisation est embarquée dans la liste, pas dans le code.
- **Un corpus figé** : la liste ne change pas au fil du notebook — les entropies calculées plus bas sont reproductibles d'une exécution à l'autre, sans aucun aléa.

### 1.1 Calcul du feedback

Implementation de la double passe. La fonction retourne un tableau de 5 entiers dans `{0,1,2}`.


In [3]:
static int[] ComputeFeedback(string guess, string answer)
{
    var feedback = new int[5];
    var answerChars = answer.ToCharArray();
    var guessChars = guess.ToCharArray();
    var used = new bool[5]; // positions du secret deja consommees par un vert/jaune

    // Passe 1 : verts (lettre correcte + bonne position). On marque la position du secret consommee.
    for (int i = 0; i < 5; i++)
    {
        if (guessChars[i] == answerChars[i])
        {
            feedback[i] = 2;
            used[i] = true;
            guessChars[i] = '\0'; // neutralise pour la passe 2
        }
    }
    // Passe 2 : jaunes (lettre presente a une autre position, premiere occurrence libre).
    for (int i = 0; i < 5; i++)
    {
        if (guessChars[i] == '\0') continue;
        for (int j = 0; j < 5; j++)
        {
            if (!used[j] && guessChars[i] == answerChars[j])
            {
                feedback[i] = 1;
                used[j] = true;
                break;
            }
        }
    }
    return feedback;
}

static string FbStr(int[] fb) => new string(fb.Select(v => "_YG"[v]).ToArray());

// Tests : couvrent l'identique, l'absence de commun, le melange, et surtout la lettre repepee.
var testCases = new (string guess, string answer, string label)[]
{
    ("CRANE", "CRANE", "Mot identique -> tout vert"),
    ("AIMER", "TABLE", "Aucune lettre commune en bonne position"),
    ("TRACE", "CRANE", "Lettres communes melangees"),
    ("ANNEE", "AIMER", "Lettre repetee dans la tentative"),
    ("ARBRE", "BRAVE", "Plusieurs lettres communes"),
};
Console.WriteLine("Tests du feedback");
Console.WriteLine(new string('=', 60));
foreach (var (g, a, label) in testCases)
{
    var fb = ComputeFeedback(g, a);
    Console.WriteLine($"  {g} vs {a} : [{FbStr(fb)}]  {string.Join(",", fb)}  -- {label}");
}


Tests du feedback


  CRANE vs CRANE : [GGGGG]  2,2,2,2,2  -- Mot identique -> tout vert


  AIMER vs TABLE : [Y__Y_]  1,0,0,1,0  -- Aucune lettre commune en bonne position


  TRACE vs CRANE : [_GGYG]  0,2,2,1,2  -- Lettres communes melangees


  ANNEE vs AIMER : [G__G_]  2,0,0,2,0  -- Lettre repetee dans la tentative


  ARBRE vs BRAVE : [YGY_G]  1,2,1,0,2  -- Plusieurs lettres communes


### Interpretation : système de feedback

La sortie teste cinq paires, de la victoire immédiate au cas le plus subtil :

- `CRANE` vs `CRANE` -> `[GGGGG]` (2,2,2,2,2) : mot identique, tout est vert ;
- `AIMER` vs `TABLE` -> `[Y__Y_]` (1,0,0,1,0) : aucune lettre commune en bonne position ;
- `TRACE` vs `CRANE` -> `[_GGYG]` (0,2,2,1,2) : lettres communes mélangées ;
- `ANNEE` vs `AIMER` -> `[G__G_]` (2,0,0,2,0) : lettre répétée dans la tentative — le `E` de la position 3 est vert et consomme le seul `E` du secret, le second `E` (final) reste gris ;
- `ARBRE` vs `BRAVE` -> `[YGY_G]` (1,2,1,0,2) : plusieurs lettres communes.

Le cas `ANNEE` est le test clé : sans la double passe (verts d'abord, en consommant les lettres du secret, puis jaunes sur les lettres restantes), le second `E` de `ANNEE` serait incorrectement marqué jaune.

### 1.2 Visualisation d'une grille Wordle (ASCII)

Le notebook Python utilisait `matplotlib` pour dessiner une grille coloree. Le twin C# restitue la même information en **ASCII** (la sortie d'un notebook pedagogique reste lisible sans dépendance graphique) ; un rendu ScottPlot est donne plus bas en option.


In [4]:
// Grille Wordle en ASCII : V = vert (bonne position), J = jaune (present, autre position), . = gris.
static void DrawGridAscii(List<string> guesses, List<int[]> feedbacks, string answer, string title = "Partie Wordle")
{
    Console.WriteLine($"  {title}  (secret : {answer})");
    Console.WriteLine("  +---+---+---+---+---+");
    int rows = Math.Max(guesses.Count, 6);
    for (int r = 0; r < rows; r++)
    {
        // ligne de lettres
        if (r < guesses.Count)
        {
            var g = guesses[r]; var fb = feedbacks[r];
            Console.Write("  |");
            for (int c = 0; c < 5; c++) Console.Write($" {g[c]} |");
            Console.WriteLine();
            Console.Write("   ");
            for (int c = 0; c < 5; c++)
            {
                string mark = fb[c] == 2 ? "V" : fb[c] == 1 ? "J" : ".";
                Console.Write($" {mark}  ");
            }
            Console.WriteLine();
        }
        else
        {
            Console.Write("  |");
            for (int c = 0; c < 5; c++) Console.Write("   |");
            Console.WriteLine();
        }
        Console.WriteLine("  +---+---+---+---+---+");
    }
}

// Demonstration sur une partie jouee a la main
var demoGuesses = new List<string> { "CRANE", "COEUR", "CRIER", "CROIX" };
var demoFeedbacks = demoGuesses.Select(g => ComputeFeedback(g, "CROIX")).ToList();
DrawGridAscii(demoGuesses, demoFeedbacks, "CROIX", "Demonstration ASCII");


  Demonstration ASCII  (secret : CROIX)


  +---+---+---+---+---+


  |

 C |

 R |

 A |

... (54 outputs left out)

   |

   |

   |

  +---+---+---+---+---+


***
## 2. Solveur par filtrage simple

Première stratégie, volontairement naive : a chaque tour, on filtre les mots incompatibles avec le feedback recu, puis on **choisit au hasard** un candidat parmi les survivants. Aucune heuristique d'information : ce solveur sert de **baseline** pour mesurer le gain apporte par le raisonnement.


In [5]:
static bool FeedbackEqual(int[] a, int[] b)
{
    for (int i = 0; i < 5; i++) if (a[i] != b[i]) return false;
    return true;
}

// Filtre : conserve les mots qui produiraient le MEME feedback si on les utilisait comme secret.
static List<string> FilterWords(List<string> candidates, string guess, int[] feedback)
{
    var result = new List<string>();
    foreach (var w in candidates)
        if (FeedbackEqual(ComputeFeedback(guess, w), feedback)) result.Add(w);
    return result;
}

class SimpleFilterSolver
{
    public readonly List<string> WordList;
    public SimpleFilterSolver(List<string> wordList) { WordList = wordList; }

    // Resout une partie ; retourne la liste des tentatives. PRNG injecte pour reproductibilite.
    public List<string> Solve(string answer, Random prng, bool verbose = false)
    {
        var candidates = new List<string>(WordList);
        var guesses = new List<string>();
        for (int attempt = 0; attempt < 6; attempt++)
        {
            if (candidates.Count == 0) break;
            string guess = candidates[prng.Next(candidates.Count)];
            guesses.Add(guess);
            var fb = ComputeFeedback(guess, answer);
            if (verbose)
                Console.WriteLine($"  Tentative {attempt+1}: {guess} -> {FbStr(fb)}  ({candidates.Count} candidats)");
            if (guess == answer) return guesses;
            candidates = FilterWords(candidates, guess, fb);
            if (verbose)
                Console.WriteLine($"    -> {candidates.Count} candidats restants");
        }
        return guesses;
    }
}

Console.WriteLine("SimpleFilterSolver defini (filtrage + tirage aleatoire).");


SimpleFilterSolver defini (filtrage + tirage aleatoire).


### Lecture — le filtrage : un prédicat de cohérence, zéro stratégie

La sortie confirme la définition de `SimpleFilterSolver` (filtrage + tirage aléatoire). Deux éléments structurent cette baseline :

- **Un prédicat de cohérence** : `FilterWords` garde un mot s'il produirait **le même feedback** que la tentative jouée (comparaison élément par élément via `FeedbackEqual`). Le mot secret est forcément dans les survivants, par construction — aucune information n'est perdue, aucune n'est exploitée au-delà de l'élimination.
- **Un choix au hasard** : le prochain mot est tiré au hasard parmi les candidats restants (`prng.Next`). C'est la baseline volontairement naïve — aucune information n'oriente le choix, le solveur subit la géométrie de l'espace des candidats.

La boucle `Solve` déroule au plus **6 tentatives** — la règle du jeu — et retourne la liste des coups joués ; le PRNG est **injecté** en paramètre (« pour reproductibilite », dit le commentaire), ce qui rend chaque partie rejouable à l'identique. La cellule suivante joue cinq parties contre des secrets fixés, en réamorçant le PRNG de façon déterministe par mot : la moyenne affichée servira de plancher aux deux solveurs suivants.

In [6]:
// Empreinte FNV-1a 32 bits : stable entre processus, BCL pure, aucune dependance NuGet.
// Necessaire car string.GetHashCode() est randomise par processus depuis .NET Core 2.1 :
// une graine derivee de ce hachage rendait les sorties du carnet non reproductibles
// d'une execution a l'autre (#23446).
static uint Fnv1a32(string s)
{
    const uint offsetBasis = 2166136261u; // 0x811C9DC5
    const uint prime = 16777619u;        // 0x01000193
    uint hash = offsetBasis;
    foreach (char c in s)
    {
        hash ^= (byte)(c & 0xFF);        // mots ASCII uppercase : un octet par caractere
        hash *= prime;
    }
    return hash;
}

// Test sur cinq mots secrets. On re-amorce le PRNG de facon deterministe par mot.
var testWords = new[] { "CRANE", "FLEUR", "MONDE", "PISTE", "VAGUE" };
Console.WriteLine("Test du solveur par filtrage simple");
Console.WriteLine(new string('=', 55));
int wins = 0; var guessCounts = new List<int>();
foreach (var word in testWords)
{
    if (!WORD_LIST.Contains(word)) { Console.WriteLine($"  {word} absent de la liste, skip"); continue; }
    // Graine deterministe par mot : empreinte FNV-1a 32 bits, STABLE entre processus.
    // (string.GetHashCode() est randomise par processus depuis .NET Core 2.1 : les sorties
    //  du carnet n'etaient pas reproductibles d'une execution a l'autre -- cf. #23446.)
    int seed = (int)(Fnv1a32(word) & 0x7fffffff);
    var solver = new SimpleFilterSolver(WORD_LIST);
    var guesses = solver.Solve(word, new Random(seed), verbose: true);
    bool won = guesses.Count > 0 && guesses[^1] == word;
    if (won) { wins++; guessCounts.Add(guesses.Count); }
    Console.WriteLine($"  => {(won ? "Gagne" : "Perdu")} en {guesses.Count} tentative(s)\n");
}
if (guessCounts.Count > 0)
    Console.WriteLine($"Moyenne (parties gagnees) : {guessCounts.Average():F1} tentatives  ({wins}/{testWords.Length} gagnees)");


Test du solveur par filtrage simple


  Tentative 1: BILAN -> ___YY  (297 candidats)


    -> 16 candidats restants


  Tentative 2: NUAGE -> Y_G_G  (16 candidats)


    -> 1 candidats restants


  Tentative 3: CRANE -> GGGGG  (1 candidats)


... (26 outputs left out)

    -> 1 candidats restants


  Tentative 4: VAGUE -> GGGGG  (1 candidats)


  => Gagne en 4 tentative(s)



Moyenne (parties gagnees) : 3.4 tentatives  (5/5 gagnees)


### Interpretation : filtrage simple

La sortie joue cinq parties et détaille la réduction des candidats à chaque tentative : `BILAN` en 3 tentatives (297 -> 16 -> 1 candidats), `PARIE` en 3 (297 -> 17 -> 1), `BARGE` en 4 (297 -> 55 -> 14 -> 2), `JEUNE` en 3 (297 -> 89 -> 2) et `CRISE` en 4 (297 -> 38 -> 3 -> 1). **Moyenne : 3.4 tentatives, 5/5 gagnées.**

Le tirage aléatoire explique les écarts d'une partie à l'autre : si le solveur élimine trop lentement les candidats, il peut épuiser ses 6 tentatives — aucune information n'oriente le choix du prochain mot. Les deux sections suivantes corrigent cela, chacune à sa manière : le solveur CSP en propageant les contraintes, le solveur par entropie en maximisant l'information.

Le detail qui rend cette section **reproductible** est la graine du tirage. Elle derive d'une empreinte **stable** -- FNV-1a 32 bits, calculee dans la cellule -- et non du hachage `string.GetHashCode()` de .NET, qui est randomise **par processus** depuis .NET Core 2.1. Avec ce dernier, deux executions du meme carnet dans deux processus distincts produisaient deux traces de tentatives differentes : les sorties committes n'etaient pas une preuve stable. L'empreinte FNV-1a, elle, ne depend que des caracteres du mot : a execution identique, trace identique.


***
## 3. Solveur CSP : propagation de contraintes

Wordle se modelise naturellement comme un **problème de satisfaction de contraintes** :

- **Variables** : les 5 positions du mot ;
- **Domaines** : pour chaque position, l'ensemble des lettres encore possibles (initialement A-Z) ;
- **Contraintes** : deduites du feedback.

  - **Vert** en position `i` pour la lettre `c` -> `domaines[i] = {c}` (la position est fixee) ;
  - **Jaune** en position `i` pour la lettre `c` -> `c` est **retiree** du domaine `i`, et `c` est declaree **requise** dans le mot ;
  - **Gris** pour la lettre `c` -> `c` est retiree des domaines (sauf si `c` est déjà requise par un jaune/vert, auquel cas on plafonne son nombre d'occurrences).

Le solveur maintient en plus des **compteurs min/max** par lettre pour gerer les repetitions. A chaque tour, il recupere les mots de la liste compatibles avec l'etat courant des domaines et des compteurs.


In [7]:
class CSPWordleSolver
{
    private readonly List<string> _wordList;
    private HashSet<char>[] _domains;        // domaine de chaque position
    private HashSet<char> _required;         // lettres qui doivent apparaitre
    private Dictionary<char,int> _minCount;  // occurrences minimales (vert + jaune)
    private Dictionary<char,int> _maxCount;  // occurrences maximales (gris plafonne)

    public CSPWordleSolver(List<string> wordList) { _wordList = wordList; Reset(); }

    public void Reset()
    {
        var all = new HashSet<char>("ABCDEFGHIJKLMNOPQRSTUVWXYZ".ToCharArray());
        _domains = new HashSet<char>[5];
        for (int i = 0; i < 5; i++) _domains[i] = new HashSet<char>(all);
        _required = new HashSet<char>();
        _minCount = new Dictionary<char,int>();
        _maxCount = new Dictionary<char,int>();
        foreach (var c in all) _maxCount[c] = 5;
    }

    // Met a jour domaines / requis / compteurs a partir d'une tentative + son feedback.
    public void AddConstraints(string guess, int[] feedback)
    {
        var greenYellow = new Dictionary<char,int>();
        var gray = new HashSet<char>();
        for (int i = 0; i < 5; i++)
        {
            char g = guess[i];
            if (feedback[i] == 2) // vert
            {
                _domains[i] = new HashSet<char> { g };
                greenYellow[g] = greenYellow.TryGetValue(g, out var v1) ? v1 + 1 : 1;
            }
            else if (feedback[i] == 1) // jaune
            {
                _domains[i].Remove(g);
                _required.Add(g);
                greenYellow[g] = greenYellow.TryGetValue(g, out var v2) ? v2 + 1 : 1;
            }
            else gray.Add(g);
        }
        // min-count : nombre exact d'occurrences connues (vert + jaune).
        foreach (var (c, n) in greenYellow) _minCount[c] = n;
        // max-count : les lettres grisees (et absentes des verts/jaunes) sont plafonnees.
        foreach (var c in gray)
        {
            if (!greenYellow.ContainsKey(c)) _maxCount[c] = 0;
            else _maxCount[c] = greenYellow[c];
        }
    }

    // Un mot est candidat s'il est compatible avec domaines + requis + compteurs.
    public List<string> GetCandidates()
    {
        var result = new List<string>();
        foreach (var w in _wordList)
        {
            bool ok = true;
            // contraintes de position (domaines)
            for (int i = 0; i < 5 && ok; i++)
                if (!_domains[i].Contains(w[i])) ok = false;
            if (!ok) continue;
            // lettres requises presentes
            foreach (var c in _required)
                if (!w.Contains(c)) { ok = false; break; }
            if (!ok) continue;
            // compteurs min/max par lettre
            var counts = w.GroupBy(ch => ch).ToDictionary(g => g.Key, g => g.Count());
            foreach (var (c, n) in counts)
            {
                if (_maxCount.TryGetValue(c, out var mx) && n > mx) { ok = false; break; }
            }
            if (!ok) continue;
            foreach (var (c, mn) in _minCount)
            {
                int n = counts.TryGetValue(c, out var v) ? v : 0;
                if (n < mn) { ok = false; break; }
            }
            if (ok) result.Add(w);
        }
        return result;
    }

    public List<string> Solve(string answer, Random prng, bool verbose = false)
    {
        Reset();
        var guesses = new List<string>();
        for (int attempt = 0; attempt < 6; attempt++)
        {
            var candidates = GetCandidates();
            if (candidates.Count == 0) break;
            string guess = candidates[prng.Next(candidates.Count)];
            guesses.Add(guess);
            var fb = ComputeFeedback(guess, answer);
            if (verbose)
                Console.WriteLine($"  Tentative {attempt+1}: {guess} -> {FbStr(fb)}  ({candidates.Count} candidats)");
            if (guess == answer) return guesses;
            AddConstraints(guess, fb);
        }
        return guesses;
    }
}

Console.WriteLine("CSPWordleSolver defini (domaines + MRV par comptage).");


CSPWordleSolver defini (domaines + MRV par comptage).


### Lecture — le Wordle comme CSP : 5 variables, 26 lettres, 3 familles de contraintes

La sortie confirme la définition de `CSPWordleSolver` (domaines + compteurs). La différence conceptuelle avec le filtrage simple vit dans la représentation :

- **5 variables** — une par position du mot — chacune avec un **domaine** initialisé aux 26 lettres (`Reset` part de `ABCDEFGHIJKLMNOPQRSTUVWXYZ` et plafonne chaque lettre à 5 occurrences, une borne qui ne mord que lorsqu'un feedback gris la réduit). Le filtrage raisonnait sur des mots complets ; le CSP raisonne sur des lettres, par position.
- **Les contraintes traduisent les couleurs** : un vert fixe la position, un jaune déclare la lettre requise (compteur min), un gris la retire des domaines ou plafonne ses occurrences (compteur max). Ces compteurs min/max gèrent les lettres répétées — le piège classique du feedback Wordle.
- **Quatre vérifications pour un candidat** : `GetCandidates` teste les domaines position par position, exige la présence des lettres requises, respecte les compteurs max, puis les compteurs min — un mot candidat satisfait les quatre conditions simultanément.

La boucle `Solve` recalcule les candidats à chaque tour (`GetCandidates` appelé dans la boucle, coiffée à 6 tentatives) : l'état dérive entièrement des feedbacks accumulés, rien n'est maintenu incrémentalement. Le mot joué reste tiré au hasard parmi les candidats compatibles : le CSP propage mieux l'information acquise, mais ne la maximise pas — c'est le rôle du solveur par entropie.

In [8]:
Console.WriteLine("Test du solveur CSP avec reduction des domaines");
Console.WriteLine(new string('=', 60));
var solverCsp = new CSPWordleSolver(WORD_LIST);
var guessesCsp = solverCsp.Solve("CRANE", new Random(42), verbose: true);
bool wonCsp = guessesCsp.Count > 0 && guessesCsp[^1] == "CRANE";
Console.WriteLine($"\n=> {(wonCsp ? "Gagne" : "Perdu")} en {guessesCsp.Count} tentative(s)");


Test du solveur CSP avec reduction des domaines


  Tentative 1: PEINE -> ___GG  (297 candidats)


  Tentative 2: CANNE -> GY_GG  (4 candidats)


  Tentative 3: CRANE -> GGGGG  (1 candidats)



=> Gagne en 3 tentative(s)


### Interpretation : solveur CSP

La sortie joue une partie contre un secret fixé : `PEINE` réduit les 297 candidats à **4** dès la première tentative, `CANNE` à 1, et `CRANE` conclut en **3 tentatives**. La propagation de contraintes fait chuter l'espace de recherche bien plus vite que le filtrage aveugle : après une seule tentative, les domaines se resserrent simultanément sur les 5 positions, et les lettres grises éliminent en bloc tous les mots qui les contiennent.

Notez toutefois que ce solveur **choisit encore au hasard** parmi les candidats compatibles : il propage mieux l'information acquise, mais ne **maximise** pas l'information de la prochaine tentative. C'est ce que réglera le solveur par entropie.

***
## 4. Théorie de l'information : choisir le mot le plus informatif

Idee cle (Cover & Thomas) : avant de jouer un mot, on peut calculer **l'information attendue** qu'il va reveler. Pour un mot `g` face a un ensemble de candidats `C`, chaque secret possible `s` produit un feedback `ComputeFeedback(g, s)`. La distribution de ces feedbacks définit une variable aleatoire ; son **entropie de Shannon**

$$H(g) = -\sum_{p} \frac{|p|}{|C|} \log_2 \frac{|p|}{|C|}$$

(où la somme porte sur les *patterns* `p` observes) mesure, en **bits**, combien le feedback de `g` reduit en moyenne l'incertitude. Plus `H(g)` est grand, plus le mot est informatif. Le mot optimal d'ouverture est celui qui maximise `H`.

Le maximum théorique est `log2(|C|)` : un mot qui repartirait uniformement le secret sur les 297 candidats en patterns equiprobables le reduirait a 1 candidat en une seule tentative.


In [9]:
static double ComputeEntropy(string guess, List<string> candidates)
{
    if (candidates.Count == 0) return 0.0;
    var patternCounts = new Dictionary<string,int>();
    foreach (var word in candidates)
    {
        var fb = ComputeFeedback(guess, word);
        string key = string.Join(",", fb); // signature du pattern
        patternCounts[key] = patternCounts.TryGetValue(key, out var v) ? v + 1 : 1;
    }
    double total = candidates.Count;
    double entropy = 0.0;
    foreach (var kv in patternCounts)
    {
        double p = kv.Value / total;
        if (p > 0) entropy -= p * Math.Log2(p);
    }
    return entropy;
}

// Classe les mots de wordPool par entropie decroissante ; wordPool = candidats par defaut.
static List<(string word, double h)> RankByEntropy(List<string> candidates, List<string> wordPool = null, int topN = 10)
{
    wordPool ??= candidates;
    var scored = new List<(string, double)>();
    foreach (var w in wordPool)
        scored.Add((w, ComputeEntropy(w, candidates)));
    scored.Sort((a, b) => b.Item2.CompareTo(a.Item2));
    return scored.Take(topN).ToList();
}

Console.WriteLine("ComputeEntropy / RankByEntropy definis.");


ComputeEntropy / RankByEntropy definis.


### Lecture — l'entropie de Shannon, du feedback au choix du mot

La sortie confirme la définition des deux fonctions. `ComputeEntropy(g, C)` évalue l'information attendue d'un mot `g` face à `C` candidats : chaque secret possible produit un feedback, la distribution de ces patterns définit une variable aléatoire dont l'entropie de Shannon mesure, en bits, la réduction moyenne d'incertitude — le nombre de questions oui/non que le feedback de `g` épargne en moyenne. `RankByEntropy` enchaîne le calcul sur tous les mots du pool et trie par entropie décroissante.

La mécanique interne est transparente : le pattern d'un secret est la signature numérique de son feedback — cinq valeurs 0, 1 ou 2 jointes par des virgules — et l'entropie est celle de la distribution de ces signatures, `-Σ p·log2(p)` sur les fréquences observées. L'espace des patterns est borné : 3 couleurs sur 5 positions, soit 3^5 = **243 patterns** — les 76 patterns distincts de `LAINE` (section 7) en occupent moins d'un tiers. Une entropie nulle signale qu'un seul pattern couvre tous les candidats : il ne reste plus rien à discriminer.

C'est le moteur unique de deux utilisations : le classement des meilleurs mots d'ouverture (section 4.1, sur la liste complète — un calcul déterministe, qui doit concorder au bit près avec le notebook Python) et le solveur par entropie (section 5, à chaque tour sur les seuls candidats restants). L'échantillon de la cellule suivante illustre l'échelle des valeurs possibles.

In [10]:
Console.WriteLine("Entropie de quelques mots (sur la liste complete)");
Console.WriteLine(new string('=', 45));
var sampleWords = new[] { "CRANE", "ADORE", "AIMER", "TABLE", "EXTRA", "FLEUR", "PISTE", "REGLE" };
foreach (var w in sampleWords)
    if (WORD_LIST.Contains(w))
        Console.WriteLine($"  {w} : H = {ComputeEntropy(w, WORD_LIST):F3} bits");

double hMax = Math.Log2(WORD_LIST.Count);
Console.WriteLine($"\nEntropie maximale theorique : log2({WORD_LIST.Count}) = {hMax:F3} bits");
Console.WriteLine($"(une seule tentative suffirait si H = {hMax:F1} bits)");


Entropie de quelques mots (sur la liste complete)


  CRANE : H = 5.445 bits


  ADORE : H = 4.954 bits


  AIMER : H = 5.027 bits


  TABLE : H = 5.082 bits


  EXTRA : H = 4.128 bits


  FLEUR : H = 4.368 bits


  PISTE : H = 4.818 bits


  REGLE : H = 4.759 bits



Entropie maximale theorique : log2(297) = 8.214 bits


(une seule tentative suffirait si H = 8.2 bits)


### Lecture — huit mots, huit entropies, un plafond théorique

La sortie évalue huit mots sur la liste complète : `CRANE` mène avec **H = 5.445 bits**, devant `TABLE` (5.082), `AIMER` (5.027), `ADORE` (4.954), `PISTE` (4.818), `REGLE` (4.759), `FLEUR` (4.368), et `EXTRA` ferme la marche à **4.128 bits**. Le plafond `log2(297) = 8.214 bits` est rappelé en dessous : aucun mot ne peut dépasser l'entropie d'une répartition uniforme des 297 secrets, et un mot atteignant 8,2 bits identifierait le secret en une seule tentative.

Même le meilleur mot de cet échantillon reste loin du plafond : le classement complet de la section suivante creuse cet écart en évaluant les 297 mots, et sa sortie chiffre la réduction attendue du vainqueur (297 -> ~7 candidats en moyenne). L'entropie ne se lit donc pas dans un mot isolé, mais dans la distribution de tous ses feedbacks possibles.

### 4.1 Les dix meilleurs mots d'ouverture

Evaluation de l'entropie des 297 mots sur la liste complete. Ce calcul est **déterministe** : il doit concorder exactement avec le notebook Python (cf. § parite ci-dessous).


In [11]:
// Top 10 des meilleurs premiers mots par entropie (point de parite Python/C#).
// Technique C548-L2 : formatter HTML integre au kernel (Microsoft.DotNet.Interactive.Formatting),
// zero dependence NuGet cote charting. SVG inline (MIME text/html) -> rend sur GitHub/nbviewer/offline
// (le CDN-script Plotly precedent rendait BLANC en rendu statique : la balise script externe ne
// s'execute pas hors-browser). Infra PlotSvg definie ici, reutilisee cell[33]. Cf #6927 / #6946.
using Microsoft.DotNet.Interactive.Formatting;
using System.IO;
using System.Text;
record PlotSvg(string Markup);
Formatter.Register(typeof(PlotSvg),
    (obj, writer) => ((TextWriter)writer).Write(((PlotSvg)obj).Markup), "text/html");

var topWords = RankByEntropy(WORD_LIST, WORD_LIST, topN: 10);
Console.WriteLine("Top 10 des meilleurs premiers mots");
Console.WriteLine(new string('-', 42));
Console.WriteLine($"{"Rang",-6}{"Mot",-10}{"Entropie (bits)",-18}");
Console.WriteLine(new string('-', 42));
int rank = 1;
foreach (var (w, h) in topWords)
{
    Console.WriteLine($"{rank,-6}{w,-10}{h,-18:F4}");
    rank++;
}

// Bar chart horizontal SVG inline : entropie (bits) par mot. L'entropie = information
// moyenne que chaque mot extrait par essai (plus grand = meilleur partitionnement).
string BuildBarSvgH(string[] labels, double[] values, string title, string xLabel) {
    const int W = 720, H = 460;
    const int mL = 110, mR = 40, mT = 50, mB = 60;
    int pW = W - mL - mR, pH = H - mT - mB;
    int n = labels.Length;
    double vMax = values.Max();
    double barH = (pH / (double)n) * 0.68;
    Func<double, double> px = v => mL + (v / vMax) * pW;
    var sb = new StringBuilder();
    sb.Append($"<svg viewBox=\"0 0 {W} {H}\" xmlns=\"http://www.w3.org/2000/svg\" style=\"font-family: -apple-system, 'Segoe UI', sans-serif; font-size: 13px;\">");
    sb.Append($"<rect x=\"0\" y=\"0\" width=\"{W}\" height=\"{H}\" fill=\"white\" stroke=\"#ddd\"/>");
    for (int i = 0; i < n; i++) {
        double yC = mT + (i + 0.5) * (pH / (double)n);
        double bw = px(values[i]) - mL;
        sb.Append($"<text x=\"{mL - 8:F1}\" y=\"{yC + 4:F1}\" fill=\"#333\" text-anchor=\"end\">{labels[i]}</text>");
        sb.Append($"<rect x=\"{mL:F1}\" y=\"{yC - barH / 2:F1}\" width=\"{bw:F1}\" height=\"{barH:F1}\" fill=\"#2a6dba\"/>");
        sb.Append($"<text x=\"{mL + bw + 6:F1}\" y=\"{yC + 4:F1}\" fill=\"#333\">{values[i]:F3}</text>");
    }
    const int gx = 5;
    for (int g = 0; g <= gx; g++) {
        double xv = vMax * g / gx;
        double xp = px(xv);
        sb.Append($"<line x1=\"{xp:F1}\" y1=\"{mT}\" x2=\"{xp:F1}\" y2=\"{mT + pH}\" stroke=\"#e8e8e8\" stroke-width=\"1\"/>");
        sb.Append($"<text x=\"{xp:F1}\" y=\"{mT + pH + 16}\" fill=\"#666\" text-anchor=\"middle\">{xv:F2}</text>");
    }
    sb.Append($"<line x1=\"{mL}\" y1=\"{mT + pH}\" x2=\"{mL + pW}\" y2=\"{mT + pH}\" stroke=\"#333\" stroke-width=\"1.5\"/>");
    sb.Append($"<text x=\"{mL + pW / 2.0}\" y=\"{H - 14}\" fill=\"#333\" text-anchor=\"middle\">{xLabel}</text>");
    sb.Append($"<text x=\"{W / 2.0}\" y=\"28\" fill=\"#222\" font-size=\"15\" font-weight=\"bold\" text-anchor=\"middle\">{title}</text>");
    sb.Append("</svg>");
    return sb.ToString();
}

{
    var words = topWords.Select(t => t.word).Reverse().ToArray();
    var ents  = topWords.Select(t => Math.Round(t.h, 4)).Reverse().ToArray();
    display(new PlotSvg(BuildBarSvgH(words, ents, "Top 10 premiers mots par entropie (H en bits)", "Entropie (bits)")));
}

var (bestWord, bestH) = topWords[0];
Console.WriteLine($"\nMeilleur premier mot : {bestWord} (H = {bestH:F4} bits)");
Console.WriteLine($"Reduction attendue : {WORD_LIST.Count} -> ~{WORD_LIST.Count / Math.Pow(2, bestH):F0} candidats en moyenne");

Top 10 des meilleurs premiers mots


------------------------------------------


Rang  Mot       Entropie (bits)   


------------------------------------------


1     LAINE     5.5115            


2     PAIRE     5.4658            


... (7 outputs left out)

10    ANCRE     5.2983            


ANCRE 5.298 PARIE 5.354 LANCE 5.366 TRAIN 5.375 TRACE 5.404 CRANE 5.445 CARTE 5.464 CLAIR 5.465 PAIRE 5.466 LAINE 5.511 0.00 1.10 2.20 3.31 4.41 5.51 Entropie (bits) Top 10 premiers mots par entropie (H en bits)


Meilleur premier mot : LAINE (H = 5.5115 bits)


Reduction attendue : 297 -> ~7 candidats en moyenne


### Interpretation : meilleurs premiers mots

La sortie classe les dix meilleurs mots d'ouverture, du rang 1 au rang 10 : `LAINE` 5.5115, `PAIRE` 5.4658, `CLAIR` 5.4650, `CARTE` 5.4638, `CRANE` 5.4451, `TRACE` 5.4044, `TRAIN` 5.3747, `LANCE` 5.3664, `PARIE` 5.3537 et `ANCRE` 5.2983 bits — avec la ligne de synthèse « Meilleur premier mot : LAINE (H = 5.5115 bits) » et la réduction attendue **297 -> ~7 candidats en moyenne**. Ces valeurs concordent **au bit près** avec le notebook Python : c'est le point de parité clé du twin — l'algorithme étant purement déterministe (aucun PRNG), les deux implémentations indépendantes C# et Python doivent produire le même classement. Le meilleur mot d'ouverture extrait environ 5,5 bits d'information, soit une réduction d'un facteur ~45 de l'ensemble des candidats en une seule tentative.

***
## 5. Solveur par entropie

Dernière stratégie : a chaque tour, **maximiser l'information attendue**. Le solveur choisit le mot (parmi les candidats encore possibles) de plus haute entropie. Cas limite : s'il ne reste qu'un ou deux candidats, l'entropie n'apporte plus rien et on les teste directement.


In [12]:
class EntropySolver
{
    private readonly List<string> _wordList;
    private readonly string _precomputedFirst;
    public EntropySolver(List<string> wordList, string precomputedFirst = null)
    { _wordList = wordList; _precomputedFirst = precomputedFirst; }

    public List<string> Solve(string answer, bool verbose = false)
    {
        var candidates = new List<string>(_wordList);
        var guesses = new List<string>();
        for (int attempt = 0; attempt < 6; attempt++)
        {
            string guess;
            if (attempt == 0 && _precomputedFirst != null) guess = _precomputedFirst;
            else if (candidates.Count <= 2) guess = candidates[0];
            else
            {
                var ranked = RankByEntropy(candidates, candidates, topN: 1);
                guess = ranked[0].word;
            }
            guesses.Add(guess);
            var fb = ComputeFeedback(guess, answer);
            double h = candidates.Count > 1 ? ComputeEntropy(guess, candidates) : 0.0;
            if (verbose)
                Console.WriteLine($"  Tentative {attempt+1}: {guess} -> {FbStr(fb)}  (H={h:F2}, {candidates.Count} candidats)");
            if (guess == answer) return guesses;
            candidates = FilterWords(candidates, guess, fb);
        }
        return guesses;
    }
}

Console.WriteLine("EntropySolver defini (max d'information a chaque tour).");


EntropySolver defini (max d'information a chaque tour).


### Lecture — entropie : un glouton sur l'information espérée

La sortie confirme la définition d'`EntropySolver` (max d'information à chaque tour). La stratégie est gloutonne : à chaque tour, jouer le mot de plus haute entropie parmi les candidats restants. Trois détails d'implémentation :

- **Premier mot pré-calculé** : l'ouverture est le meilleur mot du classement complet (`LAINE`, H = 5.5115 bits) — le recalculer à chaque partie serait du gaspillage, l'entropie du premier coup ne dépend que du corpus.
- **Deux candidats ou moins** : le solveur tente directement le premier de la liste — aucune tentative ne discrimine mieux qu'une autre, les patterns y sont quasi équiprobables.
- **Le choix reste DANS les candidats** : contrairement au vrai jeu en mode hard, ce solveur ne joue jamais un mot hors liste — c'est exactement le contraste que l'exemple guidé 1 illustre.

La boucle est coiffée à **6 tentatives** comme les deux autres solveurs, et le mode verbose affiche l'entropie du coup joué, calculée sur l'ensemble des candidats **avant** le tirage (`H=... , N candidats`) : c'est ce chiffre qui décroît tour après tour dans la sortie suivante, jusqu'à 0.00 quand un seul candidat subsiste.

Le solveur est entièrement déterministe : aucun tirage aléatoire, les parties sont rejouables à l'identique. La cellule suivante joue les cinq mêmes secrets que les deux solveurs précédents — la comparaison des moyennes, en section 6, boucle la boucle.

In [13]:
Console.WriteLine("Test du solveur par entropie (premier mot pre-calcule = meilleur)");
Console.WriteLine(new string('=', 60));
var solverEntropy = new EntropySolver(WORD_LIST, precomputedFirst: topWords[0].word);
int winsE = 0; var countsE = new List<int>();
foreach (var word in testWords)
{
    if (!WORD_LIST.Contains(word)) continue;
    var guesses = solverEntropy.Solve(word, verbose: true);
    bool won = guesses.Count > 0 && guesses[^1] == word;
    if (won) { winsE++; countsE.Add(guesses.Count); }
    Console.WriteLine($"  => {(won ? "Gagne" : "Perdu")} en {guesses.Count} tentative(s)\n");
}
if (countsE.Count > 0)
    Console.WriteLine($"Moyenne (parties gagnees) : {countsE.Average():F1} tentatives  ({winsE}/{testWords.Length} gagnees)");


Test du solveur par entropie (premier mot pre-calcule = meilleur)


  Tentative 1: LAINE -> _Y_GG  (H=5.51, 297 candidats)


  Tentative 2: ARENE -> YG_GG  (H=1.00, 2 candidats)


  Tentative 3: CRANE -> GGGGG  (H=0.00, 1 candidats)


  => Gagne en 3 tentative(s)



  Tentative 1: LAINE -> Y___Y  (H=5.51, 297 candidats)


... (13 outputs left out)

  Tentative 2: BARGE -> _G_YG  (H=3.06, 21 candidats)


  Tentative 3: VAGUE -> GGGGG  (H=0.00, 1 candidats)


  => Gagne en 3 tentative(s)



Moyenne (parties gagnees) : 3.2 tentatives  (5/5 gagnees)


### Interpretation : solveur par entropie

La sortie joue cinq parties, toutes gagnées : `LAINE` ouvre à chaque fois (H = 5.51 bits sur les 297 candidats), puis le solveur joue à chaque tour le mot de plus haute entropie parmi les survivants — `ARENE`, `GELER`, `CONTE`, `VITRE` et `BARGE` selon les parties — jusqu'à `CRANE`, `FLEUR`, `MONDE`, `PISTE` ou `VAGUE` en 3, 3, 3, 4 et 3 tentatives. **Moyenne : 3.2 tentatives, 5/5 gagnées.** L'entropie affichée à chaque tentative suit la taille de l'ensemble restant : 3.06 bits sur 21 candidats, 3.28 sur 11, 2.20 sur 9, 2.32 sur 5, 1.00 sur 2 — et **0.00 dès qu'un seul candidat subsiste**, signe qu'il ne reste plus rien à discriminer.

Contrairement aux deux premiers solveurs, celui-ci **n'utilise aucun tirage aléatoire** : son comportement est entièrement déterministe et rejouable à l'identique. Il résout la quasi-totalité des mots en peu de tentatives, parce qu'il extrait un maximum d'information à chaque coup. La comparaison chiffrée ci-dessous quantifie le gain.

***
## 6. Comparaison des trois approches

On fait jouer les trois solveurs sur un même banc de mots secrets et on compare le nombre moyen de tentatives (et le taux de reussite). Les solveurs aleatoires (filtrage simple, CSP) sont executes avec une graine fixe pour rendre le benchmark déterministe.


In [14]:
// Benchmark : moyenne de tentatives + taux de reussite sur un banc elargi.
var benchWords = WORD_LIST.Take(50).ToList(); // 50 premiers mots (deterministe, borne)

double Avg(List<int> xs) => xs.Count == 0 ? double.NaN : xs.Average();

// Filtrage simple (graine fixe)
var cSimple = new List<int>(); int wSimple = 0;
foreach (var wd in benchWords)
{
    var s = new SimpleFilterSolver(WORD_LIST);
    var gs = s.Solve(wd, new Random(42));
    if (gs.Count > 0 && gs[^1] == wd) { wSimple++; cSimple.Add(gs.Count); }
}

// CSP (graine fixe)
var cCsp = new List<int>(); int wCsp = 0;
foreach (var wd in benchWords)
{
    var s = new CSPWordleSolver(WORD_LIST);
    var gs = s.Solve(wd, new Random(42));
    if (gs.Count > 0 && gs[^1] == wd) { wCsp++; cCsp.Add(gs.Count); }
}

// Entropie (deterministe)
var cEnt = new List<int>(); int wEnt = 0;
foreach (var wd in benchWords)
{
    var s = new EntropySolver(WORD_LIST, precomputedFirst: topWords[0].word);
    var gs = s.Solve(wd);
    if (gs.Count > 0 && gs[^1] == wd) { wEnt++; cEnt.Add(gs.Count); }
}

Console.WriteLine($"Bench sur {benchWords.Count} mots secrets (graine 42 pour les solveurs aleatoires)");
Console.WriteLine(new string('=', 70));
Console.WriteLine($"{"Solveur",-28}{"Moyenne (gagnees)",-20}{"Taux reussite",-15}");
Console.WriteLine(new string('-', 70));
Console.WriteLine($"{"1. Filtrage simple",-28}{Avg(cSimple),-20:F2}{(double)wSimple/benchWords.Count,-15:P0}");
Console.WriteLine($"{"2. CSP (propagation)",-28}{Avg(cCsp),-20:F2}{(double)wCsp/benchWords.Count,-15:P0}");
Console.WriteLine($"{"3. Entropie (deterministe)",-28}{Avg(cEnt),-20:F2}{(double)wEnt/benchWords.Count,-15:P0}");


Bench sur 50 mots secrets (graine 42 pour les solveurs aleatoires)


Solveur                     Moyenne (gagnees)   Taux reussite  


----------------------------------------------------------------------


1. Filtrage simple          2.80                100 %          


2. CSP (propagation)        2.88                100 %          


3. Entropie (deterministe)  2.60                100 %          


### Interpretation : comparaison des trois approches

La sortie du benchmark sur 50 mots secrets (graine 42) donne : **filtrage simple 2.80 tentatives**, **CSP 2.88**, **entropie 2.60** — les trois à **100 %** de réussite. Le solveur par **entropie** l'emporte nettement (meilleure moyenne, déterministe). Les deux solveurs aléatoires — filtrage simple et CSP — sont **très proches** (2.80 contre 2.88 sur ce banc). Ce résultat est instructif : la propagation de contraintes du CSP ne confère pas d'avantage décisif **quand le choix final reste aléatoire**, car les deux solveurs réduisent en fin de compte au même ensemble de candidats valides (un mot est candidat ssi il est compatible avec les feedbacks, ce que les domaines CSP et la comparaison de feedback expriment de manière équivalente). L'écart observé reste dans le bruit d'un banc de 50 mots.

La leçon nette est donc ailleurs : ce n'est pas la propagation qui fait la différence, c'est **l'information**. En remplaçant le tirage aléatoire par le mot d'entropie maximale, on gagne déterministement (2.60 contre 2.80 et 2.88 en moyenne) et l'on supprime tout échec. C'est le gain pur de la théorie de l'information au-delà de la satisfaction de contraintes.

***
## 7. Distribution des feedbacks

Pour comprendre *pourquoi* un mot est informatif, on inspecte la distribution de ses feedbacks. Un bon mot d'ouverture repartit les secrets sur de **nombreux patterns peu frequents** (entropie elevee) ; un mauvais mot concentre les secrets sur peu de patterns (entropie faible).


In [15]:
// Distribution du meilleur mot d'ouverture vs un mot peu informatif.
static Dictionary<string,int> FeedbackDistribution(string guess, List<string> candidates)
{
    var counts = new Dictionary<string,int>();
    foreach (var w in candidates)
    {
        string key = string.Join(",", ComputeFeedback(guess, w));
        counts[key] = counts.TryGetValue(key, out var v) ? v + 1 : 1;
    }
    return counts;
}

string best = topWords[0].word;
// mot peu informatif : entropie minimale parmi les 50 premiers de la liste
string worst = WORD_LIST.Take(50).OrderBy(w => ComputeEntropy(w, WORD_LIST)).First();

var distBest = FeedbackDistribution(best, WORD_LIST);
var distWorst = FeedbackDistribution(worst, WORD_LIST);

Console.WriteLine($"Meilleur mot : {best} -> {distBest.Count} patterns distincts, H = {ComputeEntropy(best, WORD_LIST):F3} bits");
Console.WriteLine($"  Top-5 patterns les plus frequents : {string.Join(", ", distBest.Values.OrderByDescending(v=>v).Take(5))}");
Console.WriteLine($"Mauvais mot   : {worst} -> {distWorst.Count} patterns distincts, H = {ComputeEntropy(worst, WORD_LIST):F3} bits");
Console.WriteLine($"  Top-5 patterns les plus frequents : {string.Join(", ", distWorst.Values.OrderByDescending(v=>v).Take(5))}");
Console.WriteLine($"\nL'ecart de nombre de patterns distincts explique l'ecart d'entropie :");
Console.WriteLine($"  {best} disperse les secrets (information elevee), {worst} les concentre (information faible).");


Meilleur mot : LAINE -> 76 patterns distincts, H = 5.512 bits


  Top-5 patterns les plus frequents : 33, 21, 18, 16, 11


Mauvais mot   : APPEL -> 37 patterns distincts, H = 4.053 bits


  Top-5 patterns les plus frequents : 67, 44, 30, 16, 16



L'ecart de nombre de patterns distincts explique l'ecart d'entropie :


  LAINE disperse les secrets (information elevee), APPEL les concentre (information faible).


### Lecture — LAINE disperse, APPEL concentre : la forme de la distribution

La sortie compare deux ouvertures sur les 297 candidats. `LAINE` produit **76 patterns distincts** (H = 5.512 bits) : ses cinq plus fréquents regroupent 33, 21, 18, 16 et 11 mots — le plus gros groupe ne couvre qu'environ un neuvième du corpus. `APPEL` ne produit que **37 patterns** (H = 4.053 bits) et concentre les secrets : 67, 44, 30, 16 et 16 mots, un seul pattern en regroupant 67 à lui seul.

C'est la même mesure — l'entropie — vue par sa forme : 5,512 bits pour `LAINE` contre 4,053 bits pour `APPEL`. LAINE disperse l'incertitude sur de nombreux retours possibles (chaque feedback élimine à peu près autant de candidats), APPEL la concentre sur quelques-uns (la plupart des feedbacks sont identiques et éliminent peu). En cumulant les cinq plus grands groupes, le contraste se chiffre : 33 + 21 + 18 + 16 + 11 = **99 secrets sur 297** pour `LAINE`, contre 67 + 44 + 30 + 16 + 16 = **173 sur 297** pour `APPEL` — plus de la moitié du corpus tombe dans cinq retours possibles pour le mauvais mot. L'histogramme de la section suivante trace exactement ces deux distributions.

### 7.1 Visualisation : histogramme (distribution des feedbacks)

L'histogramme ci-dessous montre la distribution des feedbacks du **meilleur** mot d'ouverture (haute entropie) et d'un mot **peu informatif** (entropie minimale), sous forme de barres. L'intérêt pedagogique porte sur la **forme** de la distribution : le meilleur mot disperse ses feedbacks (distribution plate = information maximale, chaque feedback elimine a peu pres autant de candidats), tandis que le pire mot les concentre sur 1-2 patterns (distribution pointue = information minimale, la plupart des feedbacks sont identiques). Le rendu Plotly (technique C548-L2 : formatter HTML integre au kernel, zero dependence NuGet cote charting) remplace l'ancien rendu ASCII qui masquait la forme reelle de la distribution derriere une echelle lineaire de `#`.

In [16]:
// Histogramme (rendu SVG inline) : nombre de mots par pattern de feedback, trie par frequence decroissante.
// (PlotSvg + Formatter.Register definis en cell[21], technique C548-L2, MIME text/html.)
static string BuildBarSvgV(string[] labels, int[] values, string title, string yLabel) {
    const int W = 820, H = 470;
    const int mL = 70, mR = 40, mT = 50, mB = 135;
    int pW = W - mL - mR, pH = H - mT - mB;
    int n = labels.Length;
    if (n == 0) return "<svg viewBox=\"0 0 10 10\"></svg>";
    double vMax = values.Max();
    double barW = (pW / (double)n) * 0.72;
    Func<double, double> py = v => mT + pH - (v / vMax) * pH;
    var sb = new StringBuilder();
    sb.Append($"<svg viewBox=\"0 0 {W} {H}\" xmlns=\"http://www.w3.org/2000/svg\" style=\"font-family: -apple-system, 'Segoe UI', sans-serif; font-size: 12px;\">");
    sb.Append($"<rect x=\"0\" y=\"0\" width=\"{W}\" height=\"{H}\" fill=\"white\" stroke=\"#ddd\"/>");
    const int gy = 5;
    for (int g = 0; g <= gy; g++) {
        double yv = vMax * g / gy;
        double yp = py(yv);
        sb.Append($"<line x1=\"{mL}\" y1=\"{yp:F1}\" x2=\"{mL + pW}\" y2=\"{yp:F1}\" stroke=\"#e8e8e8\" stroke-width=\"1\"/>");
        sb.Append($"<text x=\"{mL - 8:F1}\" y=\"{yp + 4:F1}\" fill=\"#666\" text-anchor=\"end\">{(int)yv}</text>");
    }
    for (int i = 0; i < n; i++) {
        double xC = mL + (i + 0.5) * (pW / (double)n);
        double yp = py(values[i]);
        double bh = (mT + pH) - yp;
        sb.Append($"<rect x=\"{xC - barW / 2:F1}\" y=\"{yp:F1}\" width=\"{barW:F1}\" height=\"{bh:F1}\" fill=\"#3b82f6\"/>");
        sb.Append($"<text x=\"{xC:F1}\" y=\"{mT + pH + 8:F1}\" fill=\"#333\" text-anchor=\"end\" transform=\"rotate(-45 {xC:F1} {mT + pH + 8:F1})\">{labels[i]}</text>");
    }
    sb.Append($"<line x1=\"{mL}\" y1=\"{mT + pH}\" x2=\"{mL + pW}\" y2=\"{mT + pH}\" stroke=\"#333\" stroke-width=\"1.5\"/>");
    sb.Append($"<line x1=\"{mL}\" y1=\"{mT}\" x2=\"{mL}\" y2=\"{mT + pH}\" stroke=\"#333\" stroke-width=\"1.5\"/>");
    sb.Append($"<text x=\"18\" y=\"{mT + pH / 2.0:F1}\" fill=\"#333\" text-anchor=\"middle\" transform=\"rotate(-90 18 {mT + pH / 2.0:F1})\">{yLabel}</text>");
    sb.Append($"<text x=\"{W / 2.0}\" y=\"28\" fill=\"#222\" font-size=\"15\" font-weight=\"bold\" text-anchor=\"middle\">{title}</text>");
    sb.Append("</svg>");
    return sb.ToString();
}

static void HistogramSvg(string word, Dictionary<string, int> dist) {
    double h = ComputeEntropy(word, WORD_LIST);
    var vals = dist.OrderByDescending(kv => kv.Value).Take(15).ToList();
    var labels = vals.Select(kv => kv.Key).ToArray();
    var counts = vals.Select(kv => kv.Value).ToArray();
    string title = $"{word} ({dist.Count} patterns, H = {h:F3} bits)";
    display(new PlotSvg(BuildBarSvgV(labels, counts, title, "Nombre de mots")));
}

HistogramSvg(best, distBest);
HistogramSvg(worst, distWorst);

0 6 13 19 26 33 0,0,0,0,2 0,2,0,0,2 0,0,0,0,1 0,1,0,0,2 1,0,0,0,2 0,0,1,0,2 0,0,2,0,2 0,0,0,1,2 1,2,0,0,2 1,1,0,0,2 0,0,1,0,0 1,0,2,0,2 1,1,0,0,0 1,1,0,0,1 1,0,0,0,1 Nombre de mots LAINE (76 patterns, H = 5.512 bits)

0 13 26 40 53 67 0,0,0,1,0 1,0,0,1,0 0,0,0,1,1 1,0,0,0,0 0,1,0,1,0 1,0,0,1,1 0,0,0,2,0 0,0,0,0,0 2,0,0,1,0 0,0,0,2,1 1,1,0,1,0 2,0,0,0,0 1,0,0,0,1 2,0,0,2,0 1,1,0,0,0 Nombre de mots APPEL (37 patterns, H = 4.053 bits)

***
## 8. Exemples guidés et exercices

Trois exemples guidés résolus — la contribution de **dev-Clarisse** (PR étudiante [#27250](https://github.com/jsboige/CoursIA/pull/27250)) — suivis de trois exercices de remplacement, un par exemple. Chaque stub d'exercice s'exécute sans erreur (convention C.1) : il retourne un résultat neutre et imprime un message d'attente. Implémentez la solution demandée puis re-exécutez la cellule.

### Exemple guidé 1 — Meilleur second mot (crédité #27250 — dev-Clarisse)

> Résolution originale : **dev-Clarisse**, PR étudiante [#27250](https://github.com/jsboige/CoursIA/pull/27250). Reprise telle quelle ; seule la mise en forme est harmonisée avec les sections voisines.

Le solveur par entropie pré-calcule le meilleur **premier** mot. Mais après le premier feedback, le meilleur second mot **n'est pas forcément un candidat** : on peut vouloir jouer un mot hors-liste pour maximiser l'information (si cette information suffit ensuite à identifier le secret). `BestSecondGuess(candidates, wordPool)` renvoie le mot de `wordPool` (eventuellement plus large que `candidates`) maximisant l'entropie sur `candidates`.

**Approche** : parcours de `wordPool` avec `ComputeEntropy(w, candidates)`, argmax à tolérance près. Deux raffinements notables : le cas d'un seul candidat joue le secret immédiatement, et le départage à entropie égale préfère un mot **candidat** — qui peut être le secret et faire gagner tout de suite — à un mot hors-liste.

In [17]:
// Exercice 1 : retourner le mot de wordPool maximisant l'entropie sur candidates.
static string BestSecondGuess(List<string> candidates, List<string> wordPool)
{
    if (candidates == null || candidates.Count == 0) return null;
    if (candidates.Count == 1) return candidates[0];   // plus rien a discriminer : on joue le secret
    wordPool ??= candidates;

    var candidateSet = new HashSet<string>(candidates);
    string best = null;
    double bestH = -1.0;

    foreach (var w in wordPool)
    {
        double h = ComputeEntropy(w, candidates);
        // A entropie egale, on prefere un mot candidat : il peut etre le secret et gagner tout de suite.
        bool better = h > bestH + 1e-12
                   || (Math.Abs(h - bestH) <= 1e-12 && candidateSet.Contains(w) && !candidateSet.Contains(best));
        if (better)
        {
            bestH = h;
            best = w;
        }
    }
    return best;
}

Console.WriteLine("Exercice 1 : methode BestSecondGuess definie.");

// Test :
var second = BestSecondGuess(FilterWords(WORD_LIST, topWords[0].word, ComputeFeedback(topWords[0].word, "CRANE")), WORD_LIST);
Console.WriteLine($"Meilleur second mot : {second}");



Exercice 1 : methode BestSecondGuess definie.


Meilleur second mot : ARENE


### Exercice 1 : Second mot au critère des candidats restants attendus

L'entropie n'est pas la seule façon de noter un mot. Un critère plus direct : le **nombre de candidats restants attendu** après le feedback — si un mot coupe `candidates` en groupes de tailles `n1, ..., nk`, ce nombre vaut `somme(ni^2) / somme(ni)` (on tire le pattern selon sa probabilité, puis on garde son groupe). Implémentez `BestSecondGuessExpected(candidates, wordPool)` qui renvoie le mot de `wordPool` **minimisant** ce critère. Comparez avec le mot renvoyé par `BestSecondGuess` (exemple guidé 1) : la différence d'ordre vous surprend-elle ?

In [18]:
// Exercice 1 : mot de wordPool minimisant le nombre de candidats restants attendus.
static string BestSecondGuessExpected(List<string> candidates, List<string> wordPool)
{
    // TODO etudiant : pour chaque w de wordPool, calculer la distribution des feedbacks,
    // en deduire somme(ni^2)/somme(ni), et renvoyer le w qui la minimise.
    // Indice : FeedbackDistribution(w, candidates) donne le dictionnaire pattern -> effectif.
    // Etape 1 : ecrire ExpectedRemaining(w, candidates) a partir de FeedbackDistribution.
    // Etape 2 : parcourir wordPool en gardant le minimum (departage : entropie, puis mot candidat).
    Console.WriteLine("Exercice a completer -- BestSecondGuessExpected");
    return null;
}

Console.WriteLine("Exercice 1 : methode BestSecondGuessExpected definie (stub). A implementer.");

// Test (a decommenter une fois implemente) :
// var secondExp = BestSecondGuessExpected(FilterWords(WORD_LIST, topWords[0].word, ComputeFeedback(topWords[0].word, "CRANE")), WORD_LIST);
// Console.WriteLine($"Second mot (candidats restants attendus) : {secondExp}");

Exercice 1 : methode BestSecondGuessExpected definie (stub). A implementer.


### Exemple guidé 2 — Mot d'ouverture robuste (crédité #27250 — dev-Clarisse)

> Résolution originale : **dev-Clarisse**, PR étudiante [#27250](https://github.com/jsboige/CoursIA/pull/27250). Reprise telle quelle ; seule la mise en forme est harmonisée avec les sections voisines.

L'entropie mesure l'information *moyenne*. Mais un mot légèrement moins bon en moyenne peut être plus **robuste** (moins de pires cas). `WorstCaseGuess(candidates, wordPool)` optimise le **pire cas** : minimiser la taille du plus grand pattern après `g`. La cellule compare le mot optimal au pire cas avec le mot optimal en moyenne.

**Approche** : même parcours que pour l'entropie, mais la note d'un mot est la taille de son plus grand groupe (`FeedbackDistribution(...).Values.Max()`), et le départage à pire cas égal repasse par l'entropie, puis par la préférence pour un mot candidat — la cascade de départage de l'exemple guidé 1, réutilisée pour un autre critère.

In [19]:
// Exercice 2 : mot qui minimise le plus grand pattern apres feedback (minimax).
static string WorstCaseGuess(List<string> candidates, List<string> wordPool)
{
    if (candidates == null || candidates.Count == 0) return null;
    if (candidates.Count == 1) return candidates[0];
    wordPool ??= candidates;

    var candidateSet = new HashSet<string>(candidates);
    string bestWord = null;
    int bestWorst = int.MaxValue;
    double bestH = -1.0;

    foreach (var w in wordPool)
    {
        // Distribution des feedbacks : pattern -> nombre de secrets qui le produisent
        var counts = FeedbackDistribution(w, candidates);
        int worst = counts.Values.Max();   // taille du plus grand groupe = pire cas

        // Departage : a pire cas egal, la meilleure entropie, puis un mot candidat
        double h = ComputeEntropy(w, candidates);
        bool better = worst < bestWorst
                   || (worst == bestWorst && h > bestH + 1e-12)
                   || (worst == bestWorst && Math.Abs(h - bestH) <= 1e-12
                       && candidateSet.Contains(w) && !candidateSet.Contains(bestWord));
        if (better)
        {
            bestWorst = worst;
            bestH = h;
            bestWord = w;
        }
    }
    return bestWord;
}

Console.WriteLine("Exercice 2 : methode WorstCaseGuess definie.");

// Comparaison : mot optimal au pire cas vs mot optimal en moyenne (entropie)
string robustWord  = WorstCaseGuess(WORD_LIST, WORD_LIST);
string entropyWord = topWords[0].word;

foreach (var (label, w) in new[] { ("Minimax (pire cas)", robustWord), ("Entropie (moyenne)", entropyWord) })
{
    var d = FeedbackDistribution(w, WORD_LIST);
    Console.WriteLine($"{label,-20} : {w}  | pire groupe = {d.Values.Max(),4}  | H = {ComputeEntropy(w, WORD_LIST):F3} bits  | {d.Count} patterns");
}



Exercice 2 : methode WorstCaseGuess definie.


Minimax (pire cas)   : CLAIR  | pire groupe =   23  | H = 5.465 bits  | 72 patterns


Entropie (moyenne)   : LAINE  | pire groupe =   33  | H = 5.512 bits  | 76 patterns


### Exercice 2 : Mot d'ouverture au critère de la médiane

Minimax regarde le **plus grand** groupe, entropie la moyenne de tous. Entre les deux, la **médiane** des tailles de groupes ignore les extrêmes et note la robustesse « typique ». Implémentez `MedianCaseGuess(candidates, wordPool)` : le mot de `wordPool` minimisant la taille **médiane** de ses groupes de feedback (à nombre pair de groupes, moyenne des deux tailles centrales). Départage : entropie, puis mot candidat. Comparez son mot avec ceux de l'entropie et du minimax.

In [20]:
// Exercice 2 : mot qui minimise la taille mediane de ses groupes de feedback.
static string MedianCaseGuess(List<string> candidates, List<string> wordPool)
{
    // TODO etudiant : pour chaque w de wordPool, relever les tailles des groupes (FeedbackDistribution),
    // les trier, calculer la mediane, et garder le w qui la minimise.
    // Indice : counts.Values.OrderBy(x => x).ToArray(), puis mediane selon la parite.
    // Etape 1 : ecrire une fonction Mediane(int[] tailles).
    // Etape 2 : parcourir wordPool avec la meme cascade de departage que WorstCaseGuess.
    Console.WriteLine("Exercice a completer -- MedianCaseGuess");
    return null;
}

Console.WriteLine("Exercice 2 : methode MedianCaseGuess definie (stub). A implementer.");

Exercice 2 : methode MedianCaseGuess definie (stub). A implementer.


### Exemple guidé 3 — Extension à 6 lettres (crédité #27250 — dev-Clarisse)

> Résolution originale : **dev-Clarisse**, PR étudiante [#27250](https://github.com/jsboige/CoursIA/pull/27250). Reprise telle quelle ; seule la mise en forme est harmonisée avec les sections voisines.

L'extension à des mots de **6 lettres** généralise feedback, filtrage et entropie à longueur variable (`L = guess.Length`) : `ComputeFeedbackN`, `PatternKeyN`, `FilterWordsN`, `ComputeEntropyN`, puis un solveur glouton `Solve6Letters` mesuré sur une petite liste de mots de six lettres.

**Approche** : le feedback garde son algorithme en deux passes (verts d'abord, puis jaunes sur les lettres non consommées) — seule la longueur devient un paramètre ; le solveur joue le premier candidat quand il n'en reste que deux. Le plafond théorique monte avec la longueur de la grille, mais la mesure montre que la meilleure ouverture reste sous `log2(|C|)` : le corpus, pas la grille, borne l'information.

In [21]:
// Exercice 3 : adapter a 6 lettres (longueur generique L = guess.Length).

// Etape 1 : feedback a longueur variable (2 = vert, 1 = jaune, 0 = gris), meme algorithme en 2 passes.
static int[] ComputeFeedbackN(string guess, string answer)
{
    if (guess.Length != answer.Length)
        throw new ArgumentException($"Longueurs incoherentes : {guess} ({guess.Length}) vs {answer} ({answer.Length})");
    int L = guess.Length;
    var feedback = new int[L];
    var g = guess.ToCharArray();
    var used = new bool[L];

    for (int i = 0; i < L; i++)                       // passe 1 : verts
        if (g[i] == answer[i]) { feedback[i] = 2; used[i] = true; g[i] = '\0'; }

    for (int i = 0; i < L; i++)                       // passe 2 : jaunes
    {
        if (g[i] == '\0') continue;
        for (int j = 0; j < L; j++)
            if (!used[j] && g[i] == answer[j]) { feedback[i] = 1; used[j] = true; break; }
    }
    return feedback;
}

static string PatternKeyN(int[] fb) => string.Join(",", fb);

static List<string> FilterWordsN(List<string> candidates, string guess, int[] feedback)
{
    string key = PatternKeyN(feedback);
    return candidates.Where(w => PatternKeyN(ComputeFeedbackN(guess, w)) == key).ToList();
}

static double ComputeEntropyN(string guess, List<string> candidates)
{
    if (candidates.Count == 0) return 0.0;
    var counts = new Dictionary<string, int>();
    foreach (var w in candidates)
    {
        string k = PatternKeyN(ComputeFeedbackN(guess, w));
        counts[k] = counts.TryGetValue(k, out var v) ? v + 1 : 1;
    }
    double total = candidates.Count, h = 0.0;
    foreach (var c in counts.Values) { double p = c / total; h -= p * Math.Log2(p); }
    return h;
}

// Etape 2 : solveur par entropie (glouton) sur la liste 6 lettres. Retourne les tentatives.
static List<string> Solve6Letters(List<string> wordList6, string answer)
{
    if (wordList6 == null || wordList6.Count == 0) return new List<string>();
    int L = wordList6[0].Length;
    if (wordList6.Any(w => w.Length != L) || answer.Length != L)
        throw new ArgumentException($"Tous les mots doivent faire {L} lettres.");

    var candidates = new List<string>(wordList6);
    var guesses = new List<string>();
    for (int attempt = 0; attempt < 6 && candidates.Count > 0; attempt++)
    {
        string guess = candidates.Count <= 2
            ? candidates[0]
            : candidates.OrderByDescending(w => ComputeEntropyN(w, candidates)).First();
        guesses.Add(guess);
        var fb = ComputeFeedbackN(guess, answer);
        Console.WriteLine($"  Tentative {attempt + 1}: {guess} -> {string.Concat(fb)}  ({candidates.Count} candidats)");
        if (guess == answer) return guesses;
        candidates = FilterWordsN(candidates, guess, fb);
    }
    return guesses;
}

Console.WriteLine("Exercice 3 : Solve6Letters et fonctions generiques (ComputeFeedbackN, FilterWordsN, ComputeEntropyN) definies.");

// --- Mesure sur une petite liste de 6 lettres ---
var WORD_LIST_6 = new List<string> {
    "ACTEUR","AMIRAL","ANCRER","ARBRES","BALCON","BANANE","BOUGIE","BOUTON","CADEAU","CAMION",
    "CARTON","CERISE","CHAISE","CHEVAL","CITRON","COUSIN","DANGER","DESERT","DOUCHE","ECRIRE",
    "ENFANT","FLEURS","GARAGE","GLACON","JARDIN","LAPINS","LEGUME","MAISON","MARINE","MOULIN",
    "MOUTON","NUAGES","ORANGE","PAPIER","PARFUM","PIERRE","PLANTE","POMMES","RADEAU","RAISIN",
    "SALADE","SOLEIL","TABLES","TOMATE","TRAINS","VOYAGE"
}.Where(w => w.Length == 6).Distinct().ToList();


Console.WriteLine($"\nListe 6 lettres : {WORD_LIST_6.Count} mots");
Console.WriteLine($"Entropie max theorique : log2(|C|) = {Math.Log2(WORD_LIST_6.Count):F3} bits "
                + $"(plafond patterns : log2(3^6) = {6 * Math.Log2(3):F3} bits, contre {5 * Math.Log2(3):F3} pour 5 lettres)");

var best6 = WORD_LIST_6.Select(w => (w, h: ComputeEntropyN(w, WORD_LIST_6))).OrderByDescending(t => t.h).First();
Console.WriteLine($"Meilleure ouverture : {best6.w} (H = {best6.h:F3} bits)\n");

int totalGuesses = 0, solved = 0;
foreach (var secret in new[] { "JARDIN", "TOMATE", "CHEVAL", "SOLEIL" })
{
    Console.WriteLine($"Secret {secret} :");
    var gs = Solve6Letters(WORD_LIST_6, secret);
    if (gs.Count > 0 && gs[^1] == secret) { solved++; totalGuesses += gs.Count; }
}
Console.WriteLine($"\nResolus : {solved}/4, moyenne {(solved > 0 ? (double)totalGuesses / solved : 0):F2} tentatives");


Exercice 3 : Solve6Letters et fonctions generiques (ComputeFeedbackN, FilterWordsN, ComputeEntropyN) definies.



Liste 6 lettres : 46 mots


Entropie max theorique : log2(|C|) = 5.524 bits (plafond patterns : log2(3^6) = 9.510 bits, contre 7.925 pour 5 lettres)


Meilleure ouverture : MARINE (H = 5.350 bits)



Secret JARDIN :


  Tentative 1: MARINE -> 022110  (46 candidats)


... (7 outputs left out)

Secret SOLEIL :


  Tentative 1: MARINE -> 000101  (46 candidats)


  Tentative 2: SOLEIL -> 222222  (1 candidats)



Resolus : 4/4, moyenne 2.00 tentatives


### Exercice 3 : Paire d'ouverture à 6 lettres

Un seul mot d'ouverture n'épuise pas l'information disponible : une **paire** `(w1, w2)` couvre plus de lettres. Implémentez `BestOpeningPair(wordList6, budgetSecondes)` qui renvoie la paire maximisant l'entropie totale espérée sur deux coups : `H(w1) + somme des proba(pattern) * meilleure entropie d'un second mot sur le candidat filtré par ce pattern`. Attention au coût : pour chaque `w1`, chaque pattern possible filtre la liste, et chercher le meilleur `w2` sur chaque filtrat est cher — limitez le parcours (par exemple les premiers mots par entropie décroissante) pour rester dans le budget.

In [22]:
// Exercice 3 : paire d'ouverture (w1, w2) maximisant l'entropie totale esperee sur deux coups.
static (string w1, string w2, double h) BestOpeningPair(List<string> wordList6, double budgetSecondes)
{
    // TODO etudiant : pour chaque w1 candidat a l'ouverture, distribuer les patterns (ComputeFeedbackN),
    // filtrer les candidats par pattern, prendre la meilleure entropie d'un w2 sur ce filtrat,
    // et sommer pondere par la probabilite de chaque pattern.
    // Indice : ComputeEntropyN et FilterWordsN de l'exemple guide 3 font le travail par pattern.
    // Etape 1 : ecrire PairScore(w1, wordList6) = H(w1) + somme ponderee des meilleures H(w2).
    // Etape 2 : parcourir les w1 (ordre d'entropie decroissante), arreter au budget (Stopwatch).
    Console.WriteLine("Exercice a completer -- BestOpeningPair");
    return (null, null, -1.0);
}

Console.WriteLine("Exercice 3 : methode BestOpeningPair definie (stub). A implementer.");

// Test (a decommenter une fois implemente, WORD_LIST_6 est definie dans l'exemple guide 3) :
// var (w1, w2, h) = BestOpeningPair(WORD_LIST_6, 5.0);
// Console.WriteLine($"Meilleure paire : {w1} puis {w2} (H totale attendue = {h:F3} bits)");

Exercice 3 : methode BestOpeningPair definie (stub). A implementer.


***
## Conclusion

Ce twin C# a reconstruit depuis zero les trois familles de solveurs Wordle du notebook Python :

1. **Filtrage simple** (baseline aleatoire) -- mesure le plancher de performance ;
2. **CSP par propagation de contraintes** (domaines par position, lettres requises, compteurs min/max) -- exploite mieux l'information acquise ;
3. **Maximisation d'entropie** (Shannon) -- choisit a chaque tour le mot le plus informatif, déterministe.

### Parite Python / C#

Les quantites **déterministes** concordent entre les deux langages :

- Liste de mots : 297 mots identiques ;
- Entropie maximale theoretique : `log2(297) = 8.214` bits ;
- Classement top-10 des meilleurs premières tentatives (LAINE en tete) : identique au bit pres ;
- Formule de feedback (double passe, lettres repeatedes) : comportement equivalent.

Les quantites **aleatoires** (simple filtrage, solveur CSP avec tirage) différent entre C# et Python : les PRNG `System.Random` et `random.seed` ne produisent pas la même sequence. C'est attendu et **non un bug** : l'intérêt pedagogique porte sur les algorithmes et sur l'entropie (déterministe), pas sur la sequence de tirage.

En revanche, **chaque cote est reproductible d'une execution a l'autre** : la graine du tirage aleatoire derive d'une empreinte **stable**, calculee dans le carnet -- ici FNV-1a 32 bits (BCL pure). Une empreinte stable ne rend pas les deux jumeaux identiques entre eux (les PRNG diffèrent), mais elle garantit que **chaque** jumeau, rejoué, donne la meme trace qu'a sa premiere execution.

### Ponts inter-series

- **Théorie de l'information** : cf. serie Search et la formalisation de l'entropie (Cover & Thomas) ;
- **CSP** : cf. [Partie 2 (CSP)](../Part2-CSP/README.md) pour les domaines, la propagation et la consistence ;
- **App-7 Python** : la version originale avec `numpy`/`matplotlib` donne les mêmes résultats déterministes.

### Prochaines étapes

- Relire les trois exemples guidés (contribution #27250 — dev-Clarisse) et implémenter les exercices de remplacement (candidats restants attendus, médiane, paire d'ouverture) ;
- Comparer le solveur par entropie a un solveur **hybride** (minimax sur le pire cas) ;
- Revenir au [sommaire Search](../README.md) pour replacer ce notebook dans le parcours global.

## Navigation

[<< App-7 Python](App-07-Wordle-Python.ipynb) | [Index](../README.md) | [App-8 MiniZinc >>](App-08-MiniZinc-Python.ipynb)

*Marathon #4956 -- parite .NET / Python, volet Search / Applications / CSP.*
